# **SalePrice Study Notebook**

## Objectives

* Study how house attributes relate to `SalePrice`.
* Identify the variables most strongly associated with sale price.
* Create visualisations that help answer Business Requirement 1.
* Record conclusions that can be used in the Streamlit dashboard and README.

## Inputs

* `outputs/datasets/collection/house_prices_records.csv`

## Outputs

* Sale price distribution analysis.
* Correlation study between numerical variables and `SalePrice`.
* Visual plots showing relationships between selected house attributes and `SalePrice`.
* Written conclusions for dashboard and README use.

## Additional Comments

* This notebook focuses on data understanding and visual analysis.
* Data cleaning and feature engineering will be handled in later notebooks.


---

# Set working directory

Ensure the working directory is set to the project root directory.

In [ ]:
import os
from pathlib import Path

current_dir = Path.cwd()

if current_dir.name == "jupyter_notebooks":
    os.chdir(current_dir.parent)
    print("You set a new current directory")
else:
    print("Current directory is already the project root or another non-notebook directory")

Path.cwd()

# Import packages

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)

# Load data

In [ ]:
df = pd.read_csv("outputs/datasets/collection/house_prices_records.csv")
df.head()

# `SalePrice` Distribution

The target variable for the predictive analytics task is `SalePrice`. Before studying relationships with other variables, we inspect its distribution.

In [ ]:
df["SalePrice"].describe()

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(data=df, x="SalePrice", kde=True)
plt.title("Sale Price Distribution")
plt.xlabel("Sale Price")
plt.ylabel("Count")
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x="SalePrice")
plt.title("Sale Price Boxplot")
plt.xlabel("Sale Price")
plt.show()

## `SalePrice` Distribution Conclusions

* The distribution of `SalePrice` is right-skewed, with most houses sold at lower-to-mid price levels and fewer houses sold at very high prices.
* The boxplot suggests the presence of high-price outliers.
* This is expected in housing datasets, as a small number of unusually large or high-quality houses sell for much higher prices.

# Correlation Study

The following section checks how numerical variables correlate with `SalePrice`.

Pearson correlation measures linear relationships. Spearman correlation measures monotonic relationships and is less dependent on a strictly linear relationship.

In [ ]:
numeric_df = df.select_dtypes(include=["number"])

pearson_corr = numeric_df.corr(method="pearson")["SalePrice"].sort_values(ascending=False)
spearman_corr = numeric_df.corr(method="spearman")["SalePrice"].sort_values(ascending=False)

In [ ]:
pearson_corr

In [ ]:
spearman_corr

Comparison table of Pearson and Spearman numerical correlations.

In [ ]:
corr_df = pd.DataFrame({
    "pearson": pearson_corr,
    "spearman": spearman_corr
})

corr_df["pearson_abs"] = corr_df["pearson"].abs()
corr_df["spearman_abs"] = corr_df["spearman"].abs()

corr_df = corr_df.drop(index="SalePrice")
corr_df.sort_values(by="spearman_abs", ascending=False).head(10)

In [ ]:
top_corr_features = corr_df.sort_values(by="spearman_abs", ascending=False).head(10).index.to_list()
top_corr_features

In [ ]:
plt.figure(figsize=(12, 8))
sns.heatmap(
    df[top_corr_features + ["SalePrice"]].corr(method="spearman"),
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)
plt.title("Spearman Correlation Heatmap: Top Features and SalePrice")
plt.show()

## `OverallQual` and `SalePrice`

`OverallQual` rates the overall material and finish of the house. We inspect whether higher quality ratings are associated with higher sale prices.

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x="OverallQual", y="SalePrice")
plt.title("SalePrice by OverallQual")
plt.xlabel("Overall Quality")
plt.ylabel("Sale Price")
plt.show()

In [ ]:
df.groupby("OverallQual")["SalePrice"].agg(["count", "median", "mean"]).round(2)

### `OverallQual` Conclusion

The plot shows a clear relationship between `OverallQual` and `SalePrice`. Houses with higher overall quality ratings generally have higher sale prices. This suggests that overall material and finish quality is one of the strongest indicators of sale price in this dataset.

## `GrLivArea` and `SalePrice`

`GrLivArea` records above-grade living area in square feet. We inspect whether larger living areas are associated with higher sale prices.

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x="GrLivArea", y="SalePrice")
plt.title("SalePrice by Above-Grade Living Area")
plt.xlabel("Above-Grade Living Area")
plt.ylabel("Sale Price")
plt.show()

In [ ]:
df[["GrLivArea", "SalePrice"]].corr(method="spearman")

### `GrLivArea` Conclusion

The scatter plot shows that houses with larger above-grade living areas generally have higher sale prices. The relationship is positive, although there is still variation, showing that living area alone does not fully explain sale price.

## `YearBuilt` and `SalePrice`

`YearBuilt` records the original construction date of the house. In the correlation study, this variable ranked highly against `SalePrice`, especially using Spearman correlation.

This suggests that newer houses tend to be associated with higher sale prices. However, this should be interpreted as an association rather than a direct cause, because newer houses may also have other related advantages, such as better overall quality, newer remodelling, larger garages, or more modern features.

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x="YearBuilt", y="SalePrice")
plt.title("SalePrice by Year Built")
plt.xlabel("Year Built")
plt.ylabel("Sale Price")
plt.show()

In [ ]:
df_year_built = df.copy()
df_year_built["YearBuiltDecade"] = (df_year_built["YearBuilt"] // 10) * 10

plt.figure(figsize=(12, 6))
sns.boxplot(data=df_year_built, x="YearBuiltDecade", y="SalePrice")
plt.title("SalePrice by Decade Built")
plt.xlabel("Decade Built")
plt.ylabel("Sale Price")
plt.xticks(rotation=45)
plt.show()

In [ ]:
df[["YearBuilt", "SalePrice"]].corr(method="spearman")

### `YearBuilt` Conclusion

The scatter plot and decade-based boxplot suggest that newer houses generally tend to have higher sale prices. The relationship is positive, although there is still substantial variation within each period. This means that construction year is relevant to sale price, but it should be considered alongside other attributes such as overall quality, living area, garage size, basement size, and remodelling date.

## `GarageArea` and `SalePrice`

`GarageArea` records garage size in square feet. We inspect whether larger garage area is associated with higher sale prices.

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x="GarageArea", y="SalePrice")
plt.title("SalePrice by Garage Area")
plt.xlabel("Garage Area")
plt.ylabel("Sale Price")
plt.show()

### `GarageArea` Conclusion

The plot suggests a positive relationship between garage area and sale price. Houses with larger garages tend to sell for higher prices, although the relationship is less direct than the relationship between overall quality and sale price.

## `TotalBsmtSF` and `SalePrice`

`TotalBsmtSF` records total basement area in square feet. We inspect whether larger basement area is associated with higher sale prices.

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x="TotalBsmtSF", y="SalePrice")
plt.title("SalePrice by Total Basement Area")
plt.xlabel("Total Basement Area")
plt.ylabel("Sale Price")
plt.show()

### `TotalBsmtSF` Conclusion

The plot suggests a positive relationship between basement area and sale price. Houses with larger basements generally tend to sell for more, although this variable does not explain sale price by itself.

## Other Highly Correlated Numerical Features

The correlation study also identified other numerical variables with meaningful positive relationships to `SalePrice`.

These include:

* `GarageYrBlt`: the year the garage was built.
* `1stFlrSF`: first floor square feet.
* `YearRemodAdd`: remodel date, or original construction date if no remodelling or additions were made.
* `OpenPorchSF`: open porch area in square feet.
* `LotArea`: lot size in square feet.

These variables were not all explored in separate detailed subsections, but they help confirm the wider pattern that sale price is associated with house age, size, garage features, basement area, and additional property space.

In [ ]:
other_top_features = [
    "GarageYrBlt",
    "1stFlrSF",
    "YearRemodAdd",
    "OpenPorchSF",
    "LotArea"
]

corr_df.loc[other_top_features].sort_values(by="spearman_abs", ascending=False)

The following plots provide a compact visual check of these additional highly correlated features. They are included to support the correlation table, rather than to provide a separate detailed interpretation for each variable.

In [ ]:
for col in other_top_features:
    plt.figure(figsize=(10, 6))
    sns.scatterplot(data=df, x=col, y="SalePrice")
    plt.title(f"SalePrice by {col}")
    plt.xlabel(col)
    plt.ylabel("Sale Price")
    plt.show()

### Other Numerical Features Conclusion

The additional plots support the wider findings from the correlation study. Sale price tends to increase with variables related to house size, age, garage construction, remodelling, porch area, and lot area. However, these relationships vary in strength, and none of these variables alone fully explains sale price.

For the dashboard, the most useful features to highlight are `OverallQual`, `GrLivArea`, `YearBuilt`, `GarageArea`, and `TotalBsmtSF`, because they are strongly related to sale price and are relatively easy for a non-technical user to understand.

# Categorical Variable Study

Some important house attributes are categorical. The following plots inspect selected categorical variables against `SalePrice`.

## `KitchenQual` and `SalePrice`

`KitchenQual` records the quality of the kitchen.

| Code | Meaning |
| --- | --- |
| `Po` | Poor |
| `Fa` | Fair |
| `TA` | Typical/Average |
| `Gd` | Good |
| `Ex` | Excellent |

In [ ]:
df["KitchenQual"].value_counts(dropna=False)

The categories are ordered from lower to higher kitchen quality so that the plot can be read from left to right.

In [ ]:
kitchen_qual_order = ["Po", "Fa", "TA", "Gd", "Ex"]

plt.figure(figsize=(10, 6))
sns.boxplot(
    data=df,
    x="KitchenQual",
    y="SalePrice",
    order=kitchen_qual_order
)
plt.title("SalePrice by Kitchen Quality")
plt.xlabel("Kitchen Quality")
plt.ylabel("Sale Price")
plt.show()

### `KitchenQual` Conclusion

The plot suggests that houses with higher kitchen quality tend to have higher sale prices. This supports the wider finding that quality-related attributes are important sale price drivers.

## `BsmtExposure` and `SalePrice`

`BsmtExposure` refers to walkout or garden-level basement walls.

| Code | Meaning |
| --- | --- |
| `None` | No Basement |
| `No` | No Exposure |
| `Mn` | Minimum Exposure |
| `Av` | Average Exposure |
| `Gd` | Good Exposure |

In [ ]:
df["BsmtExposure"].value_counts(dropna=False)

The dataset stores houses with no basement exposure category as missing values for `BsmtExposure`. This is a visualisation step only. The formal handling of missing values will be addressed in the Data Cleaning notebook.

The categories are ordered from lower to higher basement exposure so that the plot can be read from left to right.

In [ ]:
df_plot = df.copy()
df_plot["BsmtExposure"] = df_plot["BsmtExposure"].fillna("None")

bsmt_exposure_order = ["None", "No", "Mn", "Av", "Gd"]

plt.figure(figsize=(10, 6))
sns.boxplot(
    data=df_plot,
    x="BsmtExposure",
    y="SalePrice",
    order=bsmt_exposure_order
)
plt.title("SalePrice by Basement Exposure")
plt.xlabel("Basement Exposure")
plt.ylabel("Sale Price")
plt.show()

### `BsmtExposure` Conclusion

The plot suggests that basement exposure may be associated with sale price. Houses with better basement exposure generally appear to have higher sale prices than houses with no or poor exposure.

# Correlation Summary

Save the summary of the correlation study to `.csv` files for use in the dashboard app and later notebooks.

In [ ]:
from pathlib import Path

study_output_dir = Path("outputs/datasets/collection")
study_output_dir.mkdir(parents=True, exist_ok=True)

corr_df.sort_values(by="spearman_abs", ascending=False).to_csv(
    study_output_dir / "saleprice_correlation_summary.csv"
)

In [ ]:
top_corr_features_df = corr_df.sort_values(by="spearman_abs", ascending=False).head(10)
top_corr_features_df.to_csv(study_output_dir / "saleprice_top_correlated_features.csv")
top_corr_features_df

# Conclusions and Next Steps

The sale price study found that several house attributes are strongly related to `SalePrice`.

The strongest positive relationships included:

* `OverallQual`: houses with higher overall material and finish quality tend to have higher sale prices.
* `GrLivArea`: houses with larger above-grade living areas tend to have higher sale prices.
* `YearBuilt`: newer houses tend to have higher sale prices.
* `GarageArea`: houses with larger garage areas tend to have higher sale prices.
* `TotalBsmtSF`: houses with larger basement areas tend to have higher sale prices.
* `GarageYrBlt` and `YearRemodAdd`: newer garages and more recent remodelling dates also tend to be associated with higher sale prices.

These findings help answer Business Requirement 1 by identifying the house attributes most associated with sale price.

The results suggest that sale price is especially associated with overall quality, living area, house age, garage size, basement size, and remodelling-related features.

The next notebook will focus on data cleaning so that the dataset can be prepared for feature engineering and machine learning.